In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



In [ ]:
df=pd.read_csv("/content/qoute_dataset_deep learning_rnn.csv")

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [ ]:
df.shape

(3038, 2)

In [ ]:
quotes=df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
#we have to preprocess the data first -> vectorization-> embading

In [ ]:
quotes =quotes.str.lower()#lowercse

In [ ]:
import string
translator=str.maketrans('','',string.punctuation)
quotes=quotes.apply(lambda x:x.translate(translator))


In [ ]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [ ]:
#now tokenization

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
from IPython.core.displayhook import tokenize
vocab_size=10000

tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)


In [ ]:
word_index=tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [ ]:
sequence=tokenizer.texts_to_sequences(quotes)

In [ ]:
for i in range (3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [ ]:
for i in range (3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [ ]:

X=[]
y= []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq=seq[:i]
    Output_seq=seq[i]
    X.append(input_seq)
    y.append(Output_seq)


In [ ]:
len(X)

85271

In [ ]:
len(y)

85271

In [ ]:
#now padding

In [ ]:
max_len=max([len(x) for x in X])
print(max_len)

745


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded =pad_sequences(X,maxlen=max_len,padding='pre')

In [ ]:
y=np.array(y)

In [ ]:
X_padded.shape

(85271, 745)

In [ ]:
#one hot encoding on y

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot=to_categorical(y,num_classes=vocab_size)

In [ ]:
y.shape

(85271,)

In [ ]:
y_one_hot.shape

(85271, 10000)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [ ]:
embedding_dim=50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [ ]:

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs=10
batch_size=128

In [ ]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 47s 79ms/step - accuracy: 0.0418 - loss: 6.7719 - val_accuracy: 0.0485 - val_loss: 6.6607
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 68ms/step - accuracy: 0.0608 - loss: 6.2410 - val_accuracy: 0.0762 - val_loss: 6.5430
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.0867 - loss: 5.9426 - val_accuracy: 0.0892 - val_loss: 6.3959
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 67ms/step - accuracy: 0.1040 - loss: 5.6351 - val_accuracy: 0.0999 - val_loss: 6.3916
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.1210 - loss: 5.3687 - val_accuracy: 0.1049 - val_loss: 6.4053
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.1335 - loss: 5.1231 - val_accuracy: 0.1095 - val_loss: 6.4503
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 67ms/step - accuracy: 0.1482 - loss: 4.8918 - val_accuracy: 0.1102 - val_loss: 6.5243
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.1628 - loss: 4.6717 - 

In [ ]:
epoch =100
batch_size=128

history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 37s 59ms/step - accuracy: 0.0377 - loss: 6.7657 - val_accuracy: 0.0478 - val_loss: 6.7384
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 0.0549 - loss: 6.3291 - val_accuracy: 0.0576 - val_loss: 6.6374
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 57ms/step - accuracy: 0.0751 - loss: 6.0825 - val_accuracy: 0.0811 - val_loss: 6.5431
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - accuracy: 0.0940 - loss: 5.8740 - val_accuracy: 0.0926 - val_loss: 6.5176
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 57ms/step - accuracy: 0.1079 - loss: 5.6919 - val_accuracy: 0.0968 - val_loss: 6.4951
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - accuracy: 0.1172 - loss: 5.5204 - val_accuracy: 0.1003 - val_loss: 6.4882
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 57ms/step - accuracy: 0.1262 - loss: 5.3653 - val_accuracy: 0.1040 - val_loss: 6.5388
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 43s 60ms/step - accuracy: 0.1323 - loss: 5.2241 - 

In [ ]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [ ]:
#it will save the lstm model :it will use to creat the ui of ui of this project
lstm_model.save("lstm_model.h5")

In [ ]:
#empty dictonary
index_to_word = {}
#loop through every word and its number
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [ ]:
seed_text = "what are  "
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

the


In [ ]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [ ]:
seed = "meaning of life "
generated_quote = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generated_quote)

meaning of life  is a good thing to be a man who can


In [ ]:
#to use model in ui
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)


In [ ]:
#for max length
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)